In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
zip_path = "/content/drive/MyDrive/AI folder/Week6/FruitinAmazon.zip"
extract_path = "/content/dataset"

In [4]:
import zipfile
import os

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

Dataset extracted!


In [9]:
train_dir = os.path.join(extract_path, "FruitinAmazon", "train")
# Change "train" if your folder name is different

In [10]:
print("Contents of extraction path:", os.listdir(extract_path))
class_names = sorted(os.listdir(train_dir))
print("Classes:", class_names)

Contents of extraction path: ['FruitinAmazon']
Classes: ['acai', 'cupuacu', 'graviola', 'guarana', 'pupunha', 'tucuma']


1.Check Corrupted Images

In [11]:
from PIL import Image, UnidentifiedImageError

corrupted = []

for cls in class_names:
    path = os.path.join(train_dir, cls)
    for img in os.listdir(path):
        img_path = os.path.join(path, img)
        try:
            with Image.open(img_path) as im:
                im.verify()
        except:
            corrupted.append(img_path)

print("Corrupted images:", corrupted)

Corrupted images: []


2.Create Dataset (Colab-friendly)

In [12]:
import tensorflow as tf
from tensorflow import keras

image_size = (224, 224)
batch_size = 32

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=image_size,
    batch_size=batch_size
)

Found 90 files belonging to 6 classes.
Using 72 files for training.
Using 18 files for validation.


3.x7. Optimize Performance (IMPORTANT for Colab)

In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

4. Data Augmentation (GPU optimized)

In [14]:
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2)
])

5. TASK-1 → CNN Model (Improved)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

model = Sequential([
    data_augmentation,
    Rescaling(1./255),

    Conv2D(32, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(0.25),

    Conv2D(64, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(0.25),

    Conv2D(128, 3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(0.25),

    Flatten(),

    Dense(256),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    Dense(len(class_names), activation='softmax')
])

6. Compile & Train

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 476ms/step - accuracy: 0.1944 - loss: 2.6294 - val_accuracy: 0.2222 - val_loss: 1.8148
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.6389 - loss: 0.9972 - val_accuracy: 0.1111 - val_loss: 2.0988
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.6389 - loss: 0.9248 - val_accuracy: 0.1111 - val_loss: 2.4102
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.6944 - loss: 0.9426 - val_accuracy: 0.1111 - val_loss: 2.4567
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.7778 - loss: 0.6206 - val_accuracy: 0.1111 - val_loss: 2.6090
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.8056 - loss: 0.5781 - val_accuracy: 0.1111 - val_loss: 2.8236
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8472 - loss: 0.4446 - val_accuracy: 0.1111 - val_loss: 3.0410
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8472 - loss: 0.5229 - val_accuracy: 0.1111 - val_loss

Save Model to Google Drive

In [17]:
model.save("/content/drive/MyDrive/AI folder/Week6/cnn_model.h5")

7.TASK-2 → Transfer Learning (VGG16)

In [18]:
from tensorflow.keras.applications import VGG16

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [19]:
for layer in base_model.layers:
    layer.trainable = False

In [20]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
output = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [21]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 34s 9s/step - accuracy: 0.1806 - loss: 6.4392 - val_accuracy: 0.2222 - val_loss: 2.5219
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 189ms/step - accuracy: 0.7639 - loss: 0.7705 - val_accuracy: 0.5000 - val_loss: 2.9459
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 190ms/step - accuracy: 0.8333 - loss: 0.4365 - val_accuracy: 0.6667 - val_loss: 1.8516
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 198ms/step - accuracy: 0.9583 - loss: 0.1120 - val_accuracy: 0.7778 - val_loss: 1.9557
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 257ms/step - accuracy: 0.9861 - loss: 0.0303 - val_accuracy: 0.7222 - val_loss: 2.3346
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 193ms/step - accuracy: 1.0000 - loss: 0.0070 - val_accuracy: 0.7222 - val_loss: 2.6523
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 195ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.6667 - val_loss: 2.8944
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 260ms/step - accuracy: 1.0000 - loss: 4.6063e-04 - val_accuracy: 0.6667 - val_los

In [22]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 897ms/step
              precision    recall  f1-score   support

        acai       0.00      0.00      0.00         1
     cupuacu       0.33      0.50      0.40         2
    graviola       1.00      0.67      0.80         3
     guarana       1.00      0.80      0.89         5
     pupunha       0.50      0.75      0.60         4
      tucuma       0.67      0.67      0.67         3

    accuracy                           0.67        18
   macro avg       0.58      0.56      0.56        18
weighted avg       0.70      0.67      0.67        18



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
